In [2]:
from opt_targeted_transfers import PMTTargetedTransfers
from opt_targeted_transfers import Dataset, split
from data_loaders import load_data, PATH_TO_TRAIN_DATA, PATH_TO_TEST_DATA

In [3]:
# Make train and test set
train_data = load_data(PATH_TO_TRAIN_DATA)
test_data = load_data(PATH_TO_TEST_DATA)

train_dataset = Dataset(df=train_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])
train_dataset, validation_dataset = split(train_dataset)
test_covariate_dataset = Dataset(df=test_data, outcome=None, weight='hh_wgt', covs=['hh_size', 'urban'])
test_dataset = Dataset(df=test_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])

In [4]:
tt = PMTTargetedTransfers(c_bar=2.15, transfer_value=1.0)

In [5]:
# Nuisance parameter estimation
# Fit conditional improvement regressors for different transfer values
tt.fit(train_dataset, validation_dataset)

In [6]:
# Set budget and run policy optimization step for that budget.
# Policy optimization step returns transfer amount for each unit in the test set.
# Note that budget must lie in the set of budgets used in the precomputation step.
tt.set_budget(budget=0.25)
assignments = tt.run_opt(test_covariate_dataset)

In [7]:
# Evaluate policy. 
res = tt.evaluate(test_dataset)
res

{'initial_poverty_rate': 0.6321457355538498,
 'initial_poverty_gap': 0.5455930541654876,
 'post_transfer_poverty_gap': 0.4199115415039391,
 'post_transfer_poverty_rate': 0.5361428196505152,
 'policy_cost_per_capita': 0.24977803168806176,
 'budget': 0.25,
 'policy_type': 'pmt',
 'd': 2}

In [8]:
# Can try a different budget without redoing the fit step and precomputation step.
# Note that budget must lie in the set of budgets used in the precomputation step.
# Setting the budget will clear assignments attribute.
tt.set_budget(1.0)
tt.run_opt(test_covariate_dataset)
res = tt.evaluate(test_dataset)
res

{'initial_poverty_rate': 0.6321457355538498,
 'initial_poverty_gap': 0.5455930541654876,
 'post_transfer_poverty_gap': 0.08220373278089636,
 'post_transfer_poverty_rate': 0.26207963070484036,
 'policy_cost_per_capita': 0.9999999999999991,
 'budget': 1.0,
 'policy_type': 'pmt',
 'd': 2}